In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
df=pd.read_csv(r"C:\Users\dhanu\Downloads\archive\2019-Oct.csv")
df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


In [2]:
df.shape

(42448764, 9)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42448764 entries, 0 to 42448763
Data columns (total 9 columns):
 #   Column         Dtype  
---  ------         -----  
 0   event_time     object 
 1   event_type     object 
 2   product_id     int64  
 3   category_id    int64  
 4   category_code  object 
 5   brand          object 
 6   price          float64
 7   user_id        int64  
 8   user_session   object 
dtypes: float64(1), int64(3), object(5)
memory usage: 2.8+ GB


In [4]:
df.isnull().sum()

event_time              0
event_type              0
product_id              0
category_id             0
category_code    13515609
brand             6117080
price                   0
user_id                 0
user_session            2
dtype: int64

In [6]:
df['category_code']=df['category_code'].fillna('unknown')
df['brand'] = df['brand'].fillna('Unknown')

In [7]:
df.isnull().sum()

event_time       0
event_type       0
product_id       0
category_id      0
category_code    0
brand            0
price            0
user_id          0
user_session     2
dtype: int64

In [9]:
df = df.dropna(subset=['user_session'])

In [10]:
df.isnull().sum()

event_time       0
event_type       0
product_id       0
category_id      0
category_code    0
brand            0
price            0
user_id          0
user_session     0
dtype: int64

In [12]:
df = df.drop_duplicates()

In [13]:
df['event_time']=pd.to_datetime(df['event_time'])

In [16]:
# check funnel events
df['event_time'].value_counts()

event_time
2019-10-15 08:52:00+00:00    116
2019-10-13 09:44:03+00:00    114
2019-10-24 17:07:55+00:00    113
2019-10-13 11:15:43+00:00    111
2019-10-26 09:51:31+00:00    108
                            ... 
2019-10-09 23:07:19+00:00      1
2019-10-24 22:13:02+00:00      1
2019-10-20 22:52:17+00:00      1
2019-10-09 23:07:24+00:00      1
2019-10-09 22:32:27+00:00      1
Name: count, Length: 2621538, dtype: int64

In [17]:
# create funnel metrics
total_users=df['user_id'].nunique()

In [18]:
# count each category
views = df[df['event_type'] == 'view']['user_id'].nunique()
carts = df[df['event_type'] == 'cart']['user_id'].nunique()
purchases = df[df['event_type'] == 'purchase']['user_id'].nunique()

In [19]:
# conversion rates
view_to_cart = (carts / views) * 100
cart_to_purchase = (purchases / carts) * 100
overall_conversion = (purchases / views) * 100

In [20]:
# results
print("Total Users:", total_users)
print("Views:", views)
print("Carts:", carts)
print("Purchases:", purchases)

print("View → Cart Conversion:", view_to_cart)
print("Cart → Purchase Conversion:", cart_to_purchase)
print("Overall Conversion:", overall_conversion)

Total Users: 3022290
Views: 3022130
Carts: 337117
Purchases: 347118
View → Cart Conversion: 11.154947007574128
Cart → Purchase Conversion: 102.96662583020138
Overall Conversion: 11.4858725468461


In [21]:
# funnel table
funnel = pd.DataFrame({
    'Stage': ['View', 'Cart', 'Purchase'],
    'Users': [views, carts, purchases]
})

funnel['Next Stage'] = funnel['Users'].shift(-1)
funnel['Conversion %'] = (funnel['Next Stage'] / funnel['Users']) * 100
funnel['Drop-off %'] = 100 - funnel['Conversion %']

funnel

,Stage,Users,Next Stage,Conversion %,Drop-off %
0,View,3022130,337117.0,11.154947,88.845053
1,Cart,337117,347118.0,102.966626,-2.966626
2,Purchase,347118,NaN,NaN,NaN


In [22]:
# category performance
category_analysis = df.groupby('category_code').agg(
    total_users=('user_id', 'nunique'),
    purchases=('event_type', lambda x: (x == 'purchase').sum())
).reset_index()

category_analysis['conversion_rate'] = (
    category_analysis['purchases'] / category_analysis['total_users'] * 100
)

category_analysis.sort_values(by='conversion_rate', ascending=False)

,category_code,total_users,purchases,conversion_rate
97,electronics.smartphone,1300236,337979,25.993666
90,electronics.audio.headphone,213964,30501,14.255202
26,appliances.environment.air_heater,19033,2483,13.045763
101,electronics.video.tv,170060,21561,12.678466
30,appliances.iron,29628,3652,12.326178
...,...,...,...,...
13,apparel.shoes.espadrilles,1173,0,0.000000
54,auto.accessories.anti_freeze,144,0,0.000000
18,apparel.shoes.step_ins,901,0,0.000000
87,country_yard.furniture.hammok,877,0,0.000000


In [23]:
# brand performance
brand_analysis = df.groupby('brand').agg(
    total_users=('user_id', 'nunique'),
    purchases=('event_type', lambda x: (x == 'purchase').sum())
).reset_index()

brand_analysis['conversion_rate'] = (
    brand_analysis['purchases'] / brand_analysis['total_users'] * 100
)

brand_analysis.sort_values(by='conversion_rate', ascending=False)

,brand,total_users,purchases,conversion_rate
850,dotemu,2,1,50.000000
3422,zhejiang,5,2,40.000000
574,casela,13,4,30.769231
116,alumet,4,1,25.000000
1650,kleancolor,4,1,25.000000
...,...,...,...,...
1916,marching,4,0,0.000000
1915,marcher,3,0,0.000000
1913,mapei,65,0,0.000000
439,bogates,37,0,0.000000


In [25]:
# time trend analysis
df['month'] = df['event_time'].dt.to_period('M').astype(str)

monthly = df.groupby('month').agg(
    users=('user_id', 'nunique'),
    purchases=('event_type', lambda x: (x == 'purchase').sum())
).reset_index()

monthly['conversion_rate'] = (
    monthly['purchases'] / monthly['users'] * 100
)

monthly

C:\Users\dhanu\AppData\Local\Temp\ipykernel_6664\293221951.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['month'] = df['event_time'].dt.to_period('M').astype(str)


,month,users,purchases,conversion_rate
0,2019-10,3022290,742773,24.576497


In [29]:
path = r"C:\Users\dhanu\Downloads\archive"

df.to_csv(path + "cleaned_data.csv", index=False)
funnel.to_csv(path + "funnel_data.csv", index=False)
brand_analysis.to_csv(path + "brand_analysis.csv", index=False)
category_analysis.to_csv(path + "category_analysis.csv", index=False)
monthly.to_csv(path + "monthly_analysis.csv", index=False)